# sFSC — anisotropic 3D resolution estimation

## Import the analysis and plotting functions

Import `sFSC`, the existing anisotropic resolution plotting function, and the TIFF reader. The two analysis settings are computed first, then their resolution profiles are displayed side by side. Voxel spacing is specified in **`[z, y, x]` order, in nm**.


In [1]:
from pathlib import Path
import sys
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import tifffile

# Support notebooks launched from the repository root or demos/.
repo = Path.cwd().resolve()
if not (repo / 'src' / 'sFRC.py').is_file():
    repo = repo.parent
if not (repo / 'src' / 'sFRC.py').is_file():
    raise RuntimeError('Run this notebook from the repository root or demos/ directory.')
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
example_dir = repo / 'example_data'

from src.sFSC import sFSC
from src.utils.plotting import ResolutionPlotter

## 1. Load the volume and set the voxel size

Read the 3D volume and specify the acquisition voxel spacing in **[z, y, x]** order (nm). All other analysis parameters use the defaults in `sFSC()`; the comparison below changes only `z_sampling`.


In [2]:
input_file = example_dir / 'Fig4i_1.tif'
if not input_file.is_file():
    raise FileNotFoundError(f'Add the TIFF file: {input_file}')
volume = tifffile.imread(input_file)
if volume.ndim != 3:
    raise ValueError(f'Expected a (Z,Y,X) 3D volume, got {volume.shape}')

voxel_size_nm = [60, 20, 20]  # [z, y, x]; replace with acquisition calibration.
print('Volume shape:', volume.shape)


Volume shape: (85, 2304, 2304)


## 2. Assess sensitivity to axial sampling inconsistencies

**Shared inconsistencies along the z axis may introduce artificial paraxial correlations**, leading to an overly optimistic impression of axial frequency support. In sequentially acquired volumes, inter-slice misalignment or motion can contribute to these artifacts. sFSC uses directional resolution inference (DRI) to infer the axial cutoff from non-paraxial Fourier regions rather than relying directly on the artifact-prone axial region.

As an optional sensitivity check, `z_sampling=True` forms complementary odd/even Z-plane samples for the axial analysis, doubling their effective spacing. This may reduce some shared inter-plane correlations, but also reduces axial sampling density; it does **not** necessarily improve the estimate. Compare both settings using the same input and all other parameters fixed. See [Z sampling and axial interpretation](../docs/advancedParameters.md#z-sampling-manual-control-and-axial-interpretation).

The current `sFSC()` implementation automatically plots a profile. For this notebook-only comparison, we temporarily suppress that automatic plot and call the existing `polar_plot()` utility afterward, so that **both results appear in one figure without changing `src/sFSC.py`**.

In [ ]:
# Compute both modes without creating separate automatic figures.
# This suppresses only the sFSC plot call; it does not alter either calculation.

import importlib

sfsc_module = importlib.import_module('src.sFSC')

with patch.object(sfsc_module, 'polar_plot'):
    fit_off = sFSC(volume, pixel_size=voxel_size_nm, z_sampling=False)
    fit_on = sFSC(volume, pixel_size=voxel_size_nm, z_sampling=True)

sFSC - lateral: estimating resolution...


slice-based lateral resolution estimation: 100%|████████████████████████████████████| 20/20 [00:03<00:00,  6.24plane/s]


sFSC - lateral: selected Z plane 22 (zero-based).

sFSC - lateral resolution: 182.86 nm
sFSC - axial: estimating resolution with DRI...
sFSC - DRI: fitting directional cutoffs...


directional resolution inference: 100%|████████████████████████████████████████████████| 5/5 [00:19<00:00,  3.76s/cone]

In [ ]:
# Draw the two native anisotropic profiles in one row.
# The existing polar_plot() API returns its axes but does not need an ax= argument.
fig, axes = plt.subplots(1, 2, figsize=(11, 5.1),
                         subplot_kw={'projection': 'polar'})
for ax, result, title in zip(
    axes, (fit_off, fit_on),
    ('Original axial sampling', 'Complementary Z-plane sampling')
):
    ax.set_title(title, pad=18)
    xy_nm = result.summary['lateral_resolution_nm']
    z_nm = result.summary['axial_resolution_nm']
    if np.isfinite(xy_nm) and np.isfinite(z_nm) and xy_nm > 0 and z_nm > 0:
        native_ax = ResolutionPlotter.sfsc_polar(result, show=False)
        theta, radius = native_ax.lines[0].get_data()
        ax.plot(theta, radius, color='#E67373', linewidth=2)
        ax.set_rmax(native_ax.get_rmax())
        ax.set_thetagrids(np.arange(0, 360, 45))
        ax.grid(True, alpha=0.45, linestyle='--', linewidth=0.6)
        ax.text(0.5, -0.16, f'XY: {xy_nm:.1f} nm   |   Z: {z_nm:.1f} nm',
                ha='center', va='center', transform=ax.transAxes, fontsize=9)
        plt.close(native_ax.figure)
    else:
        ax.text(0.5, 0.5, 'Resolution unavailable',
                ha='center', va='center', transform=ax.transAxes)
fig.tight_layout()
plt.show()
plt.close(fig)

A difference between the estimates indicates sensitivity to axial sampling; it does not establish which estimate is correct. Refer to [Warnings and prompts](../docs/advancedParameters.md#warnings-and-prompts) when interpreting **E_DRI** or unavailable cutoffs.
